In [85]:
import pandas as pd
from pathlib import Path

dossier_processed = Path("../data/processed")

df_inflation = pd.read_csv(dossier_processed / "inflation_senegal_features.csv")
df_taux_change = pd.read_csv(dossier_processed / "taux_change_uemoa_features.csv")
df_pib = pd.read_csv(dossier_processed / "pib_senegal_features.csv")

# On convertit la colonne period en vraie date, utile pour les modèles de série temporelle
df_inflation['period'] = pd.to_datetime(df_inflation['period'])
df_taux_change['period'] = pd.to_datetime(df_taux_change['period'])
df_pib['period'] = pd.to_datetime(df_pib['period'])

print(df_inflation.shape, df_taux_change.shape, df_pib.shape)

(27, 7) (65, 7) (65, 7)


In [86]:
# Modèle naïf : on prédit que la prochaine valeur = dernière valeur connue
# On calcule son erreur pour avoir un point de comparaison

# On sépare : tout sauf la dernière ligne pour "entraîner" (ici rien à entraîner, mais on garde la logique)
# et la dernière ligne pour tester la prédiction
train_inflation = df_inflation.iloc[:-1]
test_inflation = df_inflation.iloc[-1:]

# Prédiction naïve : la valeur prédite = dernière valeur connue avant le test
prediction_naive = train_inflation['value'].iloc[-1]
valeur_reelle = test_inflation['value'].iloc[0]

erreur_naive = abs(prediction_naive - valeur_reelle)

print("Dernière valeur connue (prédiction) :", prediction_naive)
print("Valeur réelle (2024) :", valeur_reelle)
print("Erreur absolue du modèle naïf :", erreur_naive)

Dernière valeur connue (prédiction) : 5.9
Valeur réelle (2024) : 0.8
Erreur absolue du modèle naïf : 5.1000000000000005


In [87]:
# Validation glissante : on teste le modèle naïf sur plusieurs années successives,
# pas seulement la dernière, pour avoir une mesure d'erreur plus fiable

erreurs = []

# On teste sur les 10 dernières années disponibles (à ajuster si trop ou trop peu de données)
for i in range(10, 0, -1):
    train = df_inflation.iloc[:-i]
    valeur_reelle = df_inflation['value'].iloc[-i]
    prediction_naive = train['value'].iloc[-1]

    erreur = abs(prediction_naive - valeur_reelle)
    erreurs.append(erreur)

# Erreur absolue moyenne (MAE) du modèle naïf sur ces 10 années
mae_naive = sum(erreurs) / len(erreurs)
print("Erreurs sur les 10 dernières années :", [round(e, 2) for e in erreurs])
print("Erreur absolue moyenne (MAE) du modèle naïf :", round(mae_naive, 2))

Erreurs sur les 10 dernières années : [np.float64(1.3), np.float64(0.6), np.float64(1.4), np.float64(1.7), np.float64(0.5), np.float64(1.5), np.float64(0.3), np.float64(7.5), np.float64(3.8), np.float64(5.1)]
Erreur absolue moyenne (MAE) du modèle naïf : 2.37


In [88]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings("ignore")  # SARIMA génère des avertissements techniques sans gravité

# On entraîne un SARIMA simple sur toute la série d'inflation
modele = SARIMAX(df_inflation['value'], order=(1, 1, 1))
resultat = modele.fit(disp=False)

print(resultat.summary())

                               SARIMAX Results                                
Dep. Variable:                  value   No. Observations:                   27
Model:               SARIMAX(1, 1, 1)   Log Likelihood                 -60.471
Date:                Fri, 31 Jul 2026   AIC                            126.942
Time:                        16:00:17   BIC                            130.717
Sample:                             0   HQIC                           128.029
                                 - 27                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.3271      0.400      0.818      0.413      -0.457       1.111
ma.L1         -0.9996     58.759     -0.017      0.986    -116.166     114.167
sigma2         5.5401    324.194      0.017      0.9

In [89]:
# Validation glissante pour SARIMA : on répète l'entraînement à chaque étape,
# en ajoutant à chaque fois une année de plus, pour rester réaliste (pas de triche avec le futur)

erreurs_sarima = []

for i in range(10, 0, -1):
    train = df_inflation['value'].iloc[:-i]
    valeur_reelle = df_inflation['value'].iloc[-i]

    modele = SARIMAX(train, order=(1, 1, 1))
    resultat = modele.fit(disp=False)

    prediction = resultat.forecast(steps=1).iloc[0]
    erreur = abs(prediction - valeur_reelle)
    erreurs_sarima.append(erreur)

mae_sarima = sum(erreurs_sarima) / len(erreurs_sarima)
print("Erreurs SARIMA sur les 10 dernières années :", [round(e, 2) for e in erreurs_sarima])
print("MAE SARIMA :", round(mae_sarima, 2))
print("MAE modèle naïf (référence) :", 2.37)

Erreurs SARIMA sur les 10 dernières années : [np.float64(0.82), np.float64(0.4), np.float64(0.89), np.float64(1.18), np.float64(0.26), np.float64(1.15), np.float64(0.49), np.float64(8.03), np.float64(1.5), np.float64(2.81)]
MAE SARIMA : 1.75
MAE modèle naïf (référence) : 2.37


In [90]:
# Modèle naïf sur le taux de change
erreurs_naive_tc = []
for i in range(10, 0, -1):
    train = df_taux_change['value'].iloc[:-i]
    valeur_reelle = df_taux_change['value'].iloc[-i]
    prediction_naive = train.iloc[-1]
    erreurs_naive_tc.append(abs(prediction_naive - valeur_reelle))

mae_naive_tc = sum(erreurs_naive_tc) / len(erreurs_naive_tc)

# Modèle SARIMA sur le taux de change
erreurs_sarima_tc = []
for i in range(10, 0, -1):
    train = df_taux_change['value'].iloc[:-i]
    valeur_reelle = df_taux_change['value'].iloc[-i]

    modele = SARIMAX(train, order=(1, 1, 1))
    resultat = modele.fit(disp=False)
    prediction = resultat.forecast(steps=1).iloc[0]
    erreurs_sarima_tc.append(abs(prediction - valeur_reelle))

mae_sarima_tc = sum(erreurs_sarima_tc) / len(erreurs_sarima_tc)

print("MAE naïf (taux de change) :", round(mae_naive_tc, 2))
print("MAE SARIMA (taux de change) :", round(mae_sarima_tc, 2))

MAE naïf (taux de change) : 28.47
MAE SARIMA (taux de change) : 31.26


In [91]:
# Modèle naïf sur le PIB
erreurs_naive_pib = []
for i in range(10, 0, -1):
    train = df_pib['value'].iloc[:-i]
    valeur_reelle = df_pib['value'].iloc[-i]
    prediction_naive = train.iloc[-1]
    erreurs_naive_pib.append(abs(prediction_naive - valeur_reelle))

mae_naive_pib = sum(erreurs_naive_pib) / len(erreurs_naive_pib)

# Modèle SARIMA sur le PIB
erreurs_sarima_pib = []
for i in range(10, 0, -1):
    train = df_pib['value'].iloc[:-i]
    valeur_reelle = df_pib['value'].iloc[-i]

    modele = SARIMAX(train, order=(1, 1, 1))
    resultat = modele.fit(disp=False)
    prediction = resultat.forecast(steps=1).iloc[0]
    erreurs_sarima_pib.append(abs(prediction - valeur_reelle))

mae_sarima_pib = sum(erreurs_sarima_pib) / len(erreurs_sarima_pib)

print("MAE naïf (PIB) :", round(mae_naive_pib, 2))
print("MAE SARIMA (PIB) :", round(mae_sarima_pib, 2))

MAE naïf (PIB) : 1059.06
MAE SARIMA (PIB) : 379.1


In [104]:
# Prévision finale 2026-2027 avec intervalle de confiance (PIB)
modele_final_pib = SARIMAX(df_pib['value'], order=(2, 1, 0))
resultat_final_pib = modele_final_pib.fit(disp=False)

forecast_result = resultat_final_pib.get_forecast(steps=2)  # 2026, 2027
forecast_mean = forecast_result.predicted_mean
conf_int = forecast_result.conf_int(alpha=0.05)  # IC à 95%

print("Prévisions PIB 2026-2027 :")
print(forecast_mean)
print("\nIntervalle de confiance à 95% :")
print(conf_int)

Prévisions PIB 2026-2027 :
65    21920.013686
66    23475.787834
Name: predicted_mean, dtype: float64

Intervalle de confiance à 95% :
     lower value   upper value
65  21436.874655  22403.152718
66  22559.403829  24392.171839


In [105]:
def prevoir_avec_intervalle(serie, order, steps=2, alpha=0.05):
    modele = SARIMAX(serie, order=order)
    resultat = modele.fit(disp=False)
    forecast_result = resultat.get_forecast(steps=steps)
    forecast_mean = forecast_result.predicted_mean
    conf_int = forecast_result.conf_int(alpha=alpha)
    return forecast_mean, conf_int

In [106]:
forecast_pib, conf_pib = prevoir_avec_intervalle(df_pib['value'], order=(2, 1, 0))
forecast_inflation, conf_inflation = prevoir_avec_intervalle(df_inflation['value'], order=(1, 1, 2))
# etc. pour les 7 autres, avec l'ordre SARIMA déjà trouvé pour chacun

In [92]:
import itertools

def meilleur_sarima(serie, ordres_a_tester, nb_annees_test=10):
    """Teste plusieurs combinaisons d'ordres SARIMA et retourne la meilleure selon le MAE en validation glissante."""
    resultats = []

    for ordre in ordres_a_tester:
        erreurs = []
        try:
            for i in range(nb_annees_test, 0, -1):
                train = serie.iloc[:-i]
                valeur_reelle = serie.iloc[-i]

                modele = SARIMAX(train, order=ordre)
                resultat = modele.fit(disp=False)
                prediction = resultat.forecast(steps=1).iloc[0]
                erreurs.append(abs(prediction - valeur_reelle))

            mae = sum(erreurs) / len(erreurs)
            resultats.append((ordre, mae))
        except Exception:
            # Certaines combinaisons peuvent échouer à converger, on les ignore simplement
            continue

    resultats.sort(key=lambda x: x[1])
    return resultats

# On teste toutes les combinaisons simples de p, d, q entre 0 et 2
ordres_a_tester = list(itertools.product(range(3), range(2), range(3)))
print("Nombre de combinaisons à tester :", len(ordres_a_tester))

Nombre de combinaisons à tester : 18


In [93]:
# Recherche du meilleur SARIMA pour l'inflation (peut prendre 30s à 1-2 min)
resultats_inflation = meilleur_sarima(df_inflation['value'], ordres_a_tester)

print("Top 5 des meilleures combinaisons (ordre, MAE) :")
for ordre, mae in resultats_inflation[:5]:
    print(ordre, "->", round(mae, 3))

print("\nPour rappel, MAE naïf :", 2.37, "| MAE SARIMA(1,1,1) initial :", 1.75)

Top 5 des meilleures combinaisons (ordre, MAE) :
(1, 1, 2) -> 1.418
(2, 0, 2) -> 1.427
(2, 1, 2) -> 1.548
(2, 1, 1) -> 1.67
(0, 1, 2) -> 1.75

Pour rappel, MAE naïf : 2.37 | MAE SARIMA(1,1,1) initial : 1.75


In [94]:
# Recherche du meilleur SARIMA pour le taux de change
resultats_tc = meilleur_sarima(df_taux_change['value'], ordres_a_tester)
print("Top 3 taux de change :")
for ordre, mae in resultats_tc[:3]:
    print(ordre, "->", round(mae, 3))
print("Rappel MAE naïf :", 28.47, "\n")

# Recherche du meilleur SARIMA pour le PIB
resultats_pib = meilleur_sarima(df_pib['value'], ordres_a_tester)
print("Top 3 PIB :")
for ordre, mae in resultats_pib[:3]:
    print(ordre, "->", round(mae, 3))
print("Rappel MAE naïf :", 1059.06)

Top 3 taux de change :
(1, 0, 0) -> 28.452
(0, 1, 0) -> 28.467
(2, 0, 1) -> 29.027
Rappel MAE naïf : 28.47 

Top 3 PIB :
(2, 1, 0) -> 361.606
(1, 1, 1) -> 379.103
(2, 1, 1) -> 387.606
Rappel MAE naïf : 1059.06


In [95]:
import json

for nom in ["inflation_senegal", "taux_change_uemoa", "pib_senegal"]:
    df = pd.read_csv(f"../data/processed/{nom}_features.csv")
    data = df[["period", "value"]].dropna().to_dict("records")
    with open(f"../{nom}.json", "w") as f:
        json.dump(data, f)

print("Fichiers JSON créés")

Fichiers JSON créés


In [96]:
# On explore les 2 nouveaux datasets pour trouver les codes exacts (Sénégal)
import requests
for dataset_code in ["AM_A", "AEN_A"]:
    r = requests.get(f"https://api.db.nomics.world/v22/series/BCEAO/{dataset_code}?observations=0&limit=1000")
    data = r.json()
    print(f"--- {dataset_code} ---")
    for s in data['series']['docs']:
        if 'SENEGAL' in s['series_name'].upper():
            print(s['series_code'], '-', s['series_name'])
    print()

--- AM_A ---
KKKSF1400A0AP - SENEGAL – Agregats de Monnaie - Circulation fiduciaire
KKKSF1401A0AP - SENEGAL – Agregats de Monnaie - Circulation fiduciaire - Billets et monnaies mis en circulation
KKKSF1402A0AP - SENEGAL – Agregats de Monnaie - Circulation fiduciaire - Encaisses des banques (a deduire)
KKKSF1403A0AP - SENEGAL – Agregats de Monnaie - Circulation fiduciaire - Encaisses des Tresors (a deduire)
KKKSF1404A0AP - SENEGAL – Agregats de Monnaie - Depots transferables
KKKSF1405A0AP - SENEGAL – Agregats de Monnaie - Depots transferables - BCEAO
KKKSF1406A0AP - SENEGAL – Agregats de Monnaie - Depots transferables - Autres institutions de depots (AID)
KKKSF1407A0AP - SENEGAL – Agregats de Monnaie - Depots transferables - CCP et CNE
KKKSF1408A0AP - SENEGAL – Agregats de Monnaie - M1
KKKSF1409A0AP - SENEGAL – Agregats de Monnaie - Autres depots inclus dans la masse monetaire (1)
KKKSF1410A0AP - SENEGAL – Agregats de Monnaie - Autres depots inclus dans la masse monetaire - BCEAO
KKKSF1

In [97]:
# Recherche ciblée : juste le total des actifs extérieurs nets pour le Sénégal, dans AEN_A
r = requests.get("https://api.db.nomics.world/v22/series/BCEAO/AEN_A?observations=0&limit=1000")
data_aen = r.json()

for s in data_aen['series']['docs']:
    nom = s['series_name'].upper()
    if 'SENEGAL' in nom and nom.count('-') <= 1:
        print(s['series_code'], '-', s['series_name'])

KKKSF1532A0AP - SENEGAL – Actifs exterieurs nets total 
KKKSF1533A0AP - SENEGAL – Actifs exterieurs nets total - BCEAO
KKKSF1559A0AP - SENEGAL – Actifs exterieurs nets total - Autres institutions de depots


In [98]:
# Recherche du solde commercial (balance des paiements) pour le Sénégal
r = requests.get("https://api.db.nomics.world/v22/series/BCEAO/BDP4?observations=0&limit=1000")
data_bdp = r.json()

for s in data_bdp['series']['docs']:
    nom = s['series_name'].upper()
    if 'SENEGAL' in nom and ('BALANCE' in nom or 'BIENS' in nom or 'COMMERCIALE' in nom):
        print(s['series_code'], '-', s['series_name'])

KKKSE1007A0AP - SENEGAL – 3BALANCE DES BIENS ET SERVICES
KKKSE1010A0AP - SENEGAL – 1BALANCE COMMERCIALE
KKKSE1049A0AP - SENEGAL – 2BALANCE DES SERVICES
KKKSE1310A0AP - SENEGAL – 4BALANCE DES TRANSFERTS S CONTREPARTIES
KKKSE1823A0AP - SENEGAL – 6BALANCE DES CAPITAUX PRIVES
KKKSE1835A0AP - SENEGAL – 7BALANCE DES CAPITAUX PUBLICS
KKKSE1846A0AP - SENEGAL – 8BALANCE DES CAPITAUX NON MONETAIRES


In [99]:
# On vérifie les données brutes AVANT le feature engineering, pour voir si le problème vient de la source ou du traitement
df_reserves_brut = pd.read_csv(dossier_processed.parent / "raw" / sorted([f.name for f in (dossier_processed.parent / "raw").glob("reserves_change_senegal_*.csv")])[-1])
print(df_reserves_brut.shape)
print(df_reserves_brut[['period', 'value']].tail(10))
print("Nombre de valeurs non-vides :", df_reserves_brut['value'].notna().sum())

(65, 4)
        period      value
55  2015-01-01  1232922.6
56  2016-01-01        NaN
57  2017-01-01        NaN
58  2018-01-01        NaN
59  2019-01-01        NaN
60  2020-01-01        NaN
61  2021-01-01        NaN
62  2022-01-01        NaN
63  2023-01-01        NaN
64  2024-01-01        NaN
Nombre de valeurs non-vides : 54


In [100]:
# On teste la série "ENSEMBLE UMOA" (ZZZ) au lieu de "SENEGAL" (KKK), pour les réserves de change
r = requests.get("https://api.db.nomics.world/v22/series/BCEAO/AEN_A?observations=0&limit=1000")
data_aen = r.json()

for s in data_aen['series']['docs']:
    nom = s['series_name'].upper()
    if 'UMOA' in nom and nom.count('-') <= 1:
        print(s['series_code'], '-', s['series_name'])

ZZZSF1532A0AP - ENSEMBLE UMOA – Actifs exterieurs nets total 
ZZZSF1533A0AP - ENSEMBLE UMOA – Actifs exterieurs nets total - BCEAO
ZZZSF1559A0AP - ENSEMBLE UMOA – Actifs exterieurs nets total - Autres institutions de depots


In [101]:
from dbnomics import fetch_series

df_reserves_umoa = fetch_series('BCEAO/AEN_A/ZZZSF1532A0AP')
df_reserves_umoa_valides = df_reserves_umoa[df_reserves_umoa['value'].notna()]

print("Nombre de valeurs :", len(df_reserves_umoa_valides))
print(df_reserves_umoa_valides[['period', 'value']].tail(10))

Nombre de valeurs : 0
Empty DataFrame
Columns: [period, value]
Index: []


In [102]:
for code in ["ZZZSF1533A0AP", "ZZZSF1559A0AP"]:
    df_test = fetch_series(f'BCEAO/AEN_A/{code}')
    valides = df_test[df_test['value'].notna()]
    print(code, "->", len(valides), "valeurs")

ZZZSF1533A0AP -> 0 valeurs
ZZZSF1559A0AP -> 0 valeurs


In [103]:
# On regarde toutes les années renvoyées par la Banque Mondiale, pas juste les 3 premières
for l in data_wb[1]:
    if l["value"] is not None:
        print(l["date"], "->", l["value"])

NameError: name 'data_wb' is not defined

In [ ]:
# On vérifie si toutes les valeurs sont vraiment None
print("Nombre total de lignes reçues :", len(data_wb[1]))
print("Nombre de valeurs non-None :", sum(1 for l in data_wb[1] if l["value"] is not None))
print(data_wb[1][-5:])  # les 5 dernières (les années les plus anciennes, 1960s)

NameError: name 'data_wb' is not defined

In [ ]:
for code in ["KKKSF1253A0AP", "ZZZSF1253A0AP"]:
    df_test = fetch_series(f'BCEAO/SIM/{code}')
    valides = df_test[df_test['value'].notna()]
    print(code, "->", len(valides), "valeurs")

KKKSF1253A0AP -> 54 valeurs
ZZZSF1253A0AP -> 56 valeurs


In [ ]:
df_reserves_ok = fetch_series('BCEAO/SIM/KKKSF1253A0AP')
df_reserves_valides = df_reserves_ok[df_reserves_ok['value'].notna()]
print("Fréquence :", df_reserves_ok['@frequency'].iloc[0])
print(df_reserves_valides[['period', 'value']].head(3))
print(df_reserves_valides[['period', 'value']].tail(3))

Fréquence : annual
      period    value
2 1962-01-01  15345.3
3 1963-01-01  10757.8
4 1964-01-01   8204.5
       period      value
53 2013-01-01   858465.5
54 2014-01-01  1095559.0
55 2015-01-01  1232922.6


In [ ]:
print("Dernière année avec donnée :", df_reserves_valides['period'].max())
print("Première année avec donnée :", df_reserves_valides['period'].min())

Dernière année avec donnée : 2015-01-01 00:00:00
Première année avec donnée : 1962-01-01 00:00:00


In [ ]:
df_agriculture = pd.read_csv(dossier_processed / "secteur_agriculture_senegal_features.csv")
df_industrie = pd.read_csv(dossier_processed / "secteur_industrie_senegal_features.csv")
df_services = pd.read_csv(dossier_processed / "secteur_services_senegal_features.csv")

for nom, df in [("Agriculture", df_agriculture), ("Industrie", df_industrie), ("Services", df_services)]:
    erreurs_naive = []
    for i in range(10, 0, -1):
        train = df['value'].iloc[:-i]
        valeur_reelle = df['value'].iloc[-i]
        erreurs_naive.append(abs(train.iloc[-1] - valeur_reelle))
    mae_naive = sum(erreurs_naive) / len(erreurs_naive)

    resultats = meilleur_sarima(df['value'], ordres_a_tester)
    meilleur_ordre, meilleur_mae = resultats[0]

    print(f"{nom} : MAE naïf = {round(mae_naive, 2)} | Meilleur SARIMA = {round(meilleur_mae, 2)} (ordre {meilleur_ordre})")

Agriculture : MAE naïf = 236.87 | Meilleur SARIMA = 152.77 (ordre (1, 1, 1))
Industrie : MAE naïf = 288.72 | Meilleur SARIMA = 155.1 (ordre (1, 1, 1))
Services : MAE naïf = 533.46 | Meilleur SARIMA = 211.4 (ordre (2, 1, 2))


In [113]:
df_masse_monetaire = pd.read_csv(dossier_processed / "masse_monetaire_senegal_features.csv")
df_balance_commerciale = pd.read_csv(dossier_processed / "balance_commerciale_senegal_features.csv")

In [114]:
for nom, df in [("Masse monétaire", df_masse_monetaire), ("Balance commerciale", df_balance_commerciale)]:
    erreurs_naive = []
    for i in range(10, 0, -1):
        train = df['value'].iloc[:-i]
        valeur_reelle = df['value'].iloc[-i]
        erreurs_naive.append(abs(train.iloc[-1] - valeur_reelle))
    mae_naive = sum(erreurs_naive) / len(erreurs_naive)

    resultats = meilleur_sarima(df['value'], ordres_a_tester)
    meilleur_ordre, meilleur_mae = resultats[0]

    print(f"{nom} : MAE naïf = {round(mae_naive, 2)} | Meilleur SARIMA = {round(meilleur_mae, 2)} (ordre {meilleur_ordre})")

Masse monétaire : MAE naïf = 709.77 | Meilleur SARIMA = 323.08 (ordre (2, 0, 0))
Balance commerciale : MAE naïf = 13025.6 | Meilleur SARIMA = 13025.6 (ordre (0, 1, 0))


In [116]:
forecast_pib, conf_pib = prevoir_avec_intervalle(df_pib['value'], order=(2, 1, 0))
forecast_inflation, conf_inflation = prevoir_avec_intervalle(df_inflation['value'], order=(1, 1, 2))
forecast_agriculture, conf_agriculture = prevoir_avec_intervalle(df_agriculture['value'], order=(1, 1, 1))
forecast_industrie, conf_industrie = prevoir_avec_intervalle(df_industrie['value'], order=(1, 1, 1))
forecast_services, conf_services = prevoir_avec_intervalle(df_services['value'], order=(2, 1, 2))
forecast_masse_monetaire, conf_masse_monetaire = prevoir_avec_intervalle(df_masse_monetaire['value'], order=(2, 0, 0))

print("PIB :\n", forecast_pib, "\n", conf_pib, "\n")
print("Inflation :\n", forecast_inflation, "\n", conf_inflation, "\n")
print("Agriculture :\n", forecast_agriculture, "\n", conf_agriculture, "\n")
print("Industrie :\n", forecast_industrie, "\n", conf_industrie, "\n")
print("Services :\n", forecast_services, "\n", conf_services, "\n")
print("Masse monétaire :\n", forecast_masse_monetaire, "\n", conf_masse_monetaire, "\n")

PIB :
 65    21920.013686
66    23475.787834
Name: predicted_mean, dtype: float64 
      lower value   upper value
65  21436.874655  22403.152718
66  22559.403829  24392.171839 

Inflation :
 27    2.890090
28    1.661232
Name: predicted_mean, dtype: float64 
     lower value  upper value
27    -1.256088     7.036267
28    -3.213783     6.536247 

Agriculture :
 65    4003.119651
66    4326.780632
Name: predicted_mean, dtype: float64 
     lower value  upper value
65  3823.097805  4183.141497
66  4029.766195  4623.795068 

Industrie :
 65    5585.273460
66    6016.819777
Name: predicted_mean, dtype: float64 
     lower value  upper value
65  5417.239038  5753.307882
66  5729.476260  6304.163294 

Services :
 65    12160.268765
66    12775.476687
Name: predicted_mean, dtype: float64 
      lower value   upper value
65  11860.906379  12459.631151
66  12303.577688  13247.375686 

Masse monétaire :
 21    10477.328138
22    10653.754182
Name: predicted_mean, dtype: float64 
     lower valu

In [117]:
import pandas as pd

previsions = []

def ajouter_previsions(nom_indicateur, annees, forecast, conf_int):
    for annee, valeur, lower, upper in zip(annees, forecast, conf_int.iloc[:, 0], conf_int.iloc[:, 1]):
        previsions.append({
            "indicateur": nom_indicateur,
            "annee": annee,
            "valeur_prevue": round(valeur, 2),
            "borne_basse": round(lower, 2),
            "borne_haute": round(upper, 2)
        })

annees_prevision = [2026, 2027]

ajouter_previsions("pib", annees_prevision, forecast_pib.values, conf_pib)
ajouter_previsions("inflation", annees_prevision, forecast_inflation.values, conf_inflation)
ajouter_previsions("agriculture", annees_prevision, forecast_agriculture.values, conf_agriculture)
ajouter_previsions("industrie", annees_prevision, forecast_industrie.values, conf_industrie)
ajouter_previsions("services", annees_prevision, forecast_services.values, conf_services)
ajouter_previsions("masse_monetaire", annees_prevision, forecast_masse_monetaire.values, conf_masse_monetaire)

df_previsions = pd.DataFrame(previsions)
print(df_previsions)

df_previsions.to_csv(dossier_processed / "previsions_2026_2027.csv", index=False)
df_previsions.to_json(dossier_processed / "previsions_2026_2027.json", orient="records", indent=2)

print("\nFichiers exportés dans data/processed/")

         indicateur  annee  valeur_prevue  borne_basse  borne_haute
0               pib   2026       21920.01     21436.87     22403.15
1               pib   2027       23475.79     22559.40     24392.17
2         inflation   2026           2.89        -1.26         7.04
3         inflation   2027           1.66        -3.21         6.54
4       agriculture   2026        4003.12      3823.10      4183.14
5       agriculture   2027        4326.78      4029.77      4623.80
6         industrie   2026        5585.27      5417.24      5753.31
7         industrie   2027        6016.82      5729.48      6304.16
8          services   2026       12160.27     11860.91     12459.63
9          services   2027       12775.48     12303.58     13247.38
10  masse_monetaire   2026       10477.33      9947.96     11006.70
11  masse_monetaire   2027       10653.75      9538.35     11769.16

Fichiers exportés dans data/processed/
